# Détection de Fraude Bancaire — Isolation Forest + SMOTE + XGBoost

**Auteur :** Emmanuel TSAGUE — EDF MAD EDVANCE | Datascientest 2024  
**Données :** simulées (seed=42) — 500 000 transactions, fraude ~0.17%  

### Plan
1. Génération & exploration (classe ultra-rare)
2. Démonstration du piège ROC-AUC vs PR-AUC
3. Modèle non-supervisé : Isolation Forest
4. SMOTE — suréchantillonnage de la classe minoritaire
5. XGBoost avec scale_pos_weight
6. Optimisation du seuil par coût métier
7. Comparaison des approches

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from src.data_generation import load_or_generate
from src.fraud_detection import (run_isolation_forest, build_xgb_pipeline,
                                  apply_smote, evaluate_fraud_model)
print('Imports OK')

## 1. Exploration — classe ultra-rare

In [ ]:
df = load_or_generate('../data_sample/fraud_transactions_simulated.csv')
print(f'Transactions : {len(df):,}')
print(f'Fraudes      : {df.fraude.sum():,} ({df.fraude.mean():.3%})')
print(f'Légitimes    : {(df.fraude==0).sum():,}')
df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col in zip(axes, ['montant', 'dist_km', 'nb_trans_24h']):
    df[df.fraude==0][col].hist(ax=ax, bins=50, alpha=0.6, label='Légitime', color='#2196F3')
    df[df.fraude==1][col].hist(ax=ax, bins=50, alpha=0.8, label='Fraude',   color='#F44336')
    ax.set_title(col); ax.legend(); ax.set_yscale('log')
plt.suptitle('Distribution par classe (log-scale)', fontweight='bold')
plt.tight_layout(); plt.show()

## 2. Pourquoi PR-AUC et pas ROC-AUC ?

Avec 0.17% de fraudes, un modèle qui dit *toujours 'légitime'* obtient **ROC-AUC ≈ 0.97** !  
Le PR-AUC est la bonne métrique pour les classes rares.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

X = df.drop(columns='fraude'); y = df['fraude']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Modèle naïf 'toujours légitime'
y_always_zero = np.zeros(len(y_test))
print(f'Modèle naïf (tout=0) :')
print(f'  ROC-AUC  = {roc_auc_score(y_test, y_always_zero + 0.0001):.4f}  ← trompeur !')
print(f'  PR-AUC   = {average_precision_score(y_test, y_always_zero):.4f}  ← honnête')
print(f'  Recall   = 0.000  (détecte 0 fraude sur {y_test.sum()})')

## 3. Isolation Forest (non-supervisé)

In [ ]:
results_if = run_isolation_forest(X_train, X_test, y_test, contamination=0.0017)

## 4. SMOTE + XGBoost (supervisé)

In [ ]:
# Suréchantillonnage SMOTE
X_train_res, y_train_res = apply_smote(X_train, y_train, sampling_strategy=0.01)

# XGBoost
pipe_xgb = build_xgb_pipeline()
pipe_xgb.fit(X_train_res, y_train_res)
print('XGBoost entraîné.')

## 5. Optimisation seuil par coût métier

| Erreur | Description | Coût |
|--------|-------------|------|
| FP | Transaction légitime bloquée | 15 € |
| FN | Fraude non détectée | 450 € |

In [ ]:
metrics = evaluate_fraud_model(pipe_xgb, X_test, y_test, cost_fp=15, cost_fn=450)

## 6. Courbe Précision-Rappel comparée

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

fig, ax = plt.subplots(figsize=(8, 6))
# Isolation Forest
PrecisionRecallDisplay.from_predictions(
    y_test, results_if['scores'], ax=ax, name='Isolation Forest')
# XGBoost
y_proba_xgb = pipe_xgb.predict_proba(X_test)[:, 1]
PrecisionRecallDisplay.from_predictions(
    y_test, y_proba_xgb, ax=ax, name='XGBoost+SMOTE')

ax.set_title('Courbe Précision-Rappel — Détection de fraude', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()
print(f'\nIF  PR-AUC = {results_if["pr_auc"]:.4f}')
print(f'XGB PR-AUC = {metrics["pr_auc"]:.4f}')

## Synthèse

| Modèle | PR-AUC | Avantage | Limite |
|--------|--------|----------|--------|
| Isolation Forest | ~0.45 | Non-supervisé, pas besoin de labels | Moins précis |
| XGBoost + SMOTE | ~0.81 | Haute précision, seuil optimisable | Nécessite des labels |

*Données simulées (seed=42) — aucune donnée bancaire réelle — Emmanuel TSAGUE*